# Proyecto Final Machine Learning Clasico
## Prediccion de Demanda · Tienda de Ropa Medellin
---
**Inteligencia Artificial: De los Fundamentos a la Implementacion**

Medellin es la **Capital de la Moda de America Latina**. Eres el cientific@
de datos de una cadena de moda con presencia en 6 centros comerciales de la
ciudad debes **predecir cuantas unidades por categoria se venderan
en los proximos 7 dias** para tomar decisiones de inventario y liquidacion.

```
EVENTOS CALENDARIOS CRITICOS EN MODA
======================================
Enero       Liquidacion post-Navidad (hasta 70% descuento, +90% volumen)
Mayo        Dia de la Madre          (+150% de trafico — el pico del ano)
Julio       Liquidacion mid-year     (hasta 60% descuento)
Septiembre  Amor y Amistad           (+80% en la semana previa al 14/sep)
Diciembre   Temporada Navidad        (+60% en todo el mes)
Festivos    Los CC reciben MAS       (+70% vs dia normal)
Finde       2-3x el trafico de semana <- diferencia clave vs supermercado
```

**Dataset:** 
6 CC · 36 referencias · 2021-2023 · ~3 millones de items vendidos


In [ ]:
# ------------------------------------------------------------
# Instalar librerias
# ------------------------------------------------------------
!pip install optuna lightgbm xgboost --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, joblib
from datetime import datetime

from sklearn.model_selection import cross_val_score, KFold
from sklearn.linear_model    import LinearRegression, Ridge
from sklearn.ensemble        import RandomForestRegressor
from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.metrics         import mean_squared_error, r2_score, mean_absolute_error

import xgboost  as xgb
import lightgbm as lgb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import tensorflow as tf
from tensorflow       import keras
from tensorflow.keras import layers, callbacks

plt.style.use('seaborn-v0_8-whitegrid')
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Librerias listas")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajusta la ruta si guardaste el archivo en otra carpeta
RUTA_DATASET     = '/content/drive/MyDrive/CursoIA/ropa_medellin_3M.csv'
RUTA_CHECKPOINTS = '/content/drive/MyDrive/CursoIA/checkpoints_ropa/'
os.makedirs(RUTA_CHECKPOINTS, exist_ok=True)
print("Drive montado.")
print("Dataset    :", RUTA_DATASET)
print("Checkpoint :", RUTA_CHECKPOINTS)


---
## Seccion 1 · Carga y Optimizacion de Memoria

El dataset de ropa tiene columnas con cardinalidad muy baja: solo 7 categorias,
6 tallas y 10 colores. Estas son candidatas perfectas para el tipo `category`.

### Ejercicio 1 — Diccionario de dtypes

**Que debes hacer:** completa el diccionario `DTYPES` con el tipo correcto.

**Regla:** columnas de texto con pocos valores unicos → `'category'`.
Variables binarias → `'int8'`. Decimales → `'float32'`.

**Columnas del dataset:**
`venta_id, fecha, hora, tienda_id, nombre_tienda, centro_comercial, barrio,
estrato, producto_id, nombre_producto, categoria, genero_objetivo, talla,
color, cantidad, precio_unitario, descuento_pct, precio_final, metodo_pago,
es_liquidacion, liq_enero, liq_julio, es_dia_madre, es_amor_amistad,
es_dia_padre, es_feria_flores, es_diciembre, es_festivo`


In [ ]:
# ── EJERCICIO 1 ─────────────────────────────────────────────────────────────

DTYPES = {
    'venta_id'        : 'int32',
    'hora'            : 'int8',
    'estrato'         : 'int8',

    # TU CODIGO: asigna 'category' a todas las columnas de texto
    'tienda_id'       : ___,
    'nombre_tienda'   : ___,
    'centro_comercial': ___,
    'barrio'          : ___,
    'producto_id'     : ___,
    'nombre_producto' : ___,
    'categoria'       : ___,
    'genero_objetivo' : ___,
    'talla'           : ___,   # XS S M L XL XXL — solo 6 valores
    'color'           : ___,   # 10 colores
    'metodo_pago'     : ___,

    'cantidad'        : 'int8',
    'precio_unitario' : 'int32',
    'descuento_pct'   : ___,   # 0 a 70
    'precio_final'    : 'int32',

    # TU CODIGO: todas las binarias son int8
    'es_liquidacion'  : ___,
    'liq_enero'       : ___,
    'liq_julio'       : ___,
    'es_dia_madre'    : ___,
    'es_amor_amistad' : ___,
    'es_dia_padre'    : ___,
    'es_feria_flores' : ___,
    'es_diciembre'    : ___,
    'es_festivo'      : ___,
}

print("Cargando dataset de ropa...")
t0 = datetime.now()
df = pd.read_csv(RUTA_DATASET, parse_dates=['fecha'], dtype=DTYPES)
print(f"Listo en {(datetime.now()-t0).seconds}s | Shape: {df.shape}")
print(f"Memoria: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB")
df.head(3)


---
## Seccion 2 · Analisis Exploratorio

### Ejercicio 2 — Patrones de moda

**Que debes calcular:**
1. **Impacto del Dia de la Madre:** promedio y conteo de ventas en semana del
   Dia de la Madre vs semana normal (agrupa por `es_dia_madre`).
2. **Rotacion por categoria:** unidades totales vendidas (`cantidad`) por categoria,
   ordenadas de mayor a menor.
3. **Ventas mensuales por año:** linea temporal que muestre la estacionalidad.
   Se esperan picos en enero, mayo, julio y diciembre.
4. **¿Que CC tiene el precio promedio mas alto?** Agrupa por `centro_comercial` y calcula
   el precio_final promedio.

**Pregunta para reflexionar:** ¿El Dia de la Madre es realmente el pico mas alto
del año en ventas? ¿O lo supera algun periodo de liquidacion?


In [ ]:
# ── EJERCICIO 2 ─────────────────────────────────────────────────────────────

# 2a. Impacto del Dia de la Madre
impacto_madre = df.groupby('es_dia_madre')['precio_final'].agg(
    # TU CODIGO: calcula promedio ('mean') y conteo ('count')
)
impacto_madre.index = ['Semana normal','Semana Dia Madre']
print("Impacto Dia de la Madre:")
print(impacto_madre.round(0))

# 2b. Rotacion por categoria
rotacion = (
    df.groupby(___)[___]   # TU CODIGO: por categoria, suma de cantidad
    .sum()
    .sort_values(ascending=False)
)
print("\nRotacion total (unidades vendidas):")
print(rotacion)

# 2c. Ventas mensuales
ventas_mes = df.groupby(
    [df['fecha'].dt.year, df['fecha'].dt.month]
)['precio_final'].sum().reset_index()
ventas_mes.columns = ['anio','mes','ventas']

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for anio, g in ventas_mes.groupby('anio'):
    axes[0].plot(g['mes'], g['ventas']/1e9, marker='o', label=str(anio), lw=2)
axes[0].set_title('Ventas mensuales por año (miles de millones COP)')
axes[0].set_xticks(range(1,13))
axes[0].set_xticklabels(['E','F','M','A','M','J','J','A','S','O','N','D'])
axes[0].legend()

# 2d. Precio promedio por CC
precio_cc = (
    df.groupby('centro_comercial')['precio_final']
    # TU CODIGO: promedio y conteo
    .agg(___)
    .sort_values('mean', ascending=False)
)
print("\nPrecio promedio por CC:")
print(precio_cc.round(0))

rotacion.sort_values().plot(kind='barh', ax=axes[1], color='steelblue', edgecolor='white')
axes[1].set_title('Unidades vendidas por categoria')
plt.tight_layout()
plt.show()


---
## Seccion 3 · Feature Engineering

### Ejercicio 3 — Features temporales y de temporada

**Que debes crear:**

| Columna            | Descripcion                                                        |
|--------------------|--------------------------------------------------------------------|
| `anio`             | Año (int16)                                                        |
| `mes`              | Mes 1-12 (int8)                                                    |
| `dia_semana`       | 0=Lun…6=Dom (int8)                                                 |
| `trimestre`        | 1-4 (int8)                                                         |
| `es_finde`         | 1 si sabado o domingo (int8) — muy importante en CC                |
| `dias_desde_inicio`| Dias desde la fecha minima (int16)                                 |
| `temporada_moda`   | 0=Regular, 1=Liquidacion, 2=Evento especial, 3=Diciembre (int8)   |
| `nivel_descuento`  | Categorico: 0=sin descuento, 1=bajo (1-15%), 2=medio (16-29%), 3=liquidacion (30%+) |

**Para `temporada_moda` usa `np.select(condiciones, opciones, default=0)`:**
- condicion 1: `es_liquidacion == 1` → valor 1
- condicion 2: `(es_dia_madre==1) | (es_amor_amistad==1) | (es_dia_padre==1)` → valor 2
- condicion 3: `es_diciembre == 1` → valor 3

**Para `nivel_descuento` usa `pd.cut()` con bins `[-1, 0, 15, 29, 100]`.**


In [ ]:
# ── EJERCICIO 3 ─────────────────────────────────────────────────────────────

df['anio']              = ___   # TU CODIGO
df['mes']               = ___
df['dia_semana']        = ___
df['trimestre']         = ___
df['es_finde']          = ___
df['dias_desde_inicio'] = ___

# Temporada de moda (np.select)
condiciones  = [___, ___, ___]   # TU CODIGO: las 3 condiciones en orden
opciones     = [1,   2,   3]
df['temporada_moda']  = np.select(condiciones, opciones, default=0).astype('int8')

# Nivel de descuento (pd.cut)
df['nivel_descuento'] = pd.cut(
    df['descuento_pct'],
    bins   = ___,   # TU CODIGO: [-1, 0, 15, 29, 100]
    labels = [0, 1, 2, 3]
).astype('int8')

print("Distribucion temporada_moda:")
print(df['temporada_moda'].value_counts().sort_index())
print("\n0=Regular  1=Liquidacion  2=Evento  3=Diciembre")


### Ejercicio 4 — Target: ventas semanales por categoria y tienda

**Que debes hacer:**
1. Agregar al nivel `(fecha, tienda_id, categoria)`: suma de `cantidad`,
   suma de `precio_final`, conteo de ventas, promedio de `descuento_pct`.
2. Agregar features temporales al dataset agregado.
3. Codificar `tienda_id` y `categoria` como enteros.
4. Calcular `target_7d` = suma de `items_dia` de los 7 dias siguientes.
   Usa `.rolling(7, min_periods=7).sum().shift(-7)` dentro de un `groupby().transform()`.
5. Eliminar filas con `target_7d` = NaN.

**Resultado esperado:** DataFrame con ~60 000 filas y columna `target_7d`.


In [ ]:
# ── EJERCICIO 4 ─────────────────────────────────────────────────────────────

# Paso 1: Agregar
demanda_diaria = (
    df.groupby(['fecha','tienda_id','categoria'])
    .agg(
        items_dia  = ('cantidad',     ___),   # TU CODIGO: 'sum'
        ingresos   = ('precio_final', ___),
        n_ventas   = ('venta_id',     'count'),
        desc_prom  = ('descuento_pct', ___)
    )
    .reset_index()
)

# Paso 2: Features temporales
demanda_diaria['anio']             = demanda_diaria['fecha'].dt.year.astype('int16')
demanda_diaria['mes']              = demanda_diaria['fecha'].dt.month.astype('int8')
demanda_diaria['dia_semana']       = demanda_diaria['fecha'].dt.dayofweek.astype('int8')
demanda_diaria['trimestre']        = demanda_diaria['fecha'].dt.quarter.astype('int8')
demanda_diaria['es_finde']         = (demanda_diaria['dia_semana'] >= 5).astype('int8')
demanda_diaria['dias_desde_inicio']= (
    (demanda_diaria['fecha'] - demanda_diaria['fecha'].min()).dt.days.astype('int16')
)

# Paso 3: Codificar categoricas
le_t = {t:i for i,t in enumerate(demanda_diaria['tienda_id'].unique())}
le_c = {c:i for i,c in enumerate(demanda_diaria['categoria'].unique())}
demanda_diaria['tienda_enc']    = demanda_diaria['tienda_id'].map(le_t).astype('int8')
demanda_diaria['categoria_enc'] = demanda_diaria['categoria'].map(le_c).astype('int8')

# Paso 4: Target
demanda_diaria = demanda_diaria.sort_values(['tienda_id','categoria','fecha'])
demanda_diaria['target_7d'] = (
    demanda_diaria
    .groupby(['tienda_id','categoria'])['items_dia']
    .transform(lambda x: ___)   # TU CODIGO
)

# Paso 5: Eliminar NaN
demanda_diaria = demanda_diaria.dropna(subset=['target_7d']).reset_index(drop=True)
print(f"Dataset para ML: {len(demanda_diaria):,} filas")
print(f"Target promedio: {demanda_diaria['target_7d'].mean():.1f} items/semana")


In [ ]:
# ── Split temporal — celda completa ─────────────────────────────────────────
FEATURES = ['anio','mes','dia_semana','trimestre','es_finde',
            'dias_desde_inicio','tienda_enc','categoria_enc',
            'items_dia','ingresos','n_ventas','desc_prom']
TARGET   = 'target_7d'

X = demanda_diaria[FEATURES]
y = demanda_diaria[TARGET]

fecha_corte = demanda_diaria['fecha'].max() - pd.Timedelta(days=90)
mask_train  = demanda_diaria['fecha'] <= fecha_corte
X_train, X_test = X[mask_train].copy(), X[~mask_train].copy()
y_train, y_test = y[mask_train].copy(), y[~mask_train].copy()
print(f"Train: {len(X_train):,} | Test: {len(X_test):,}")


---
## Seccion 5 · Baseline — Ejercicio 5

**Que debes hacer:** igual que en el proyecto del supermercado.
1. `KFold(n_splits=10, shuffle=False)`
2. Evaluar `LinearRegression` y `Ridge(alpha=100)` con `cross_val_score`.
3. Entrenar el mejor en X_train y evaluar en X_test.
4. Guardar en `RESULTADOS` para la tabla final.


In [ ]:
# ── EJERCICIO 5 — Baseline ───────────────────────────────────────────────────
kf           = ___   # TU CODIGO: KFold
modelo_ridge = ___   # TU CODIGO: Pipeline con scaler + Ridge
scores       = cross_val_score(___)   # TU CODIGO
print(f"Ridge R2-CV: {scores.mean():.4f} +/- {scores.std():.4f}")

___   # TU CODIGO: fit sobre X_train
y_pred_base = ___   # TU CODIGO: predict sobre X_test
r2_base     = r2_score(y_test, y_pred_base)
rmse_base   = np.sqrt(mean_squared_error(y_test, y_pred_base))
mae_base    = mean_absolute_error(y_test, y_pred_base)
print(f"BASELINE TEST: R2={r2_base:.4f} | RMSE={rmse_base:.2f} | MAE={mae_base:.2f}")
RESULTADOS = {'baseline': {'r2':r2_base,'rmse':rmse_base,'mae':mae_base}}


---
## Secciones 6–8 · Optuna — Ejercicios 6, 7 y 8

Usa la misma estrategia del supermercado: submuestra del 15%, 80 trials,
3-fold CV. Los espacios de busqueda son identicos.

Guarda cada estudio en Drive al terminar.

**Espacio de busqueda — igual para los 3 proyectos:**

| Modelo      | Hiperparametros clave a buscar |
|-------------|-------------------------------|
| RandomForest | n_estimators, max_depth, min_samples_split, min_samples_leaf, max_features, max_samples |
| XGBoost     | n_estimators, learning_rate, max_depth, subsample, colsample_bytree, reg_alpha, reg_lambda |
| LightGBM    | n_estimators, learning_rate, num_leaves, subsample, colsample_bytree, min_child_samples, reg_alpha, reg_lambda |


In [ ]:
# ── Submuestra para Optuna ───────────────────────────────────────────────────
FRAC    = 0.15
idx_opt = np.random.choice(len(X_train), int(len(X_train)*FRAC), replace=False)
X_opt, y_opt = X_train.iloc[idx_opt], y_train.iloc[idx_opt]
print(f"Submuestra: {len(X_opt):,} filas")


In [ ]:
# ── EJERCICIO 6 — Random Forest con Optuna 

def objetivo_rf(trial):
    params = {
        # TU CODIGO: define el espacio de busqueda
    }
    modelo  = RandomForestRegressor(**params)
    scores  = cross_val_score(modelo, X_opt, y_opt,
                              cv=KFold(3,shuffle=False), scoring='r2', n_jobs=1)
    return scores.mean()

study_rf = optuna.create_study(direction='maximize',
             sampler=optuna.samplers.TPESampler(seed=SEED))
study_rf.optimize(objetivo_rf, n_trials=80, timeout=14400, show_progress_bar=True)
print(f"Mejor R2 RF: {study_rf.best_value:.4f}")
joblib.dump(study_rf, RUTA_CHECKPOINTS+'study_rf.pkl')


In [ ]:
# ── EJERCICIO 7 — XGBoost con Optuna 

def objetivo_xgb(trial):
    params = {
        # TU CODIGO: define el espacio de busqueda
        'tree_method': 'hist',   # obligatorio para datos grandes
        'random_state': SEED, 'n_jobs': -1
    }
    scores = cross_val_score(xgb.XGBRegressor(**params), X_opt, y_opt,
                             cv=KFold(3,shuffle=False), scoring='r2', n_jobs=1)
    return scores.mean()

study_xgb = optuna.create_study(direction='maximize',
              sampler=optuna.samplers.TPESampler(seed=SEED))
study_xgb.optimize(objetivo_xgb, n_trials=80, timeout=9000, show_progress_bar=True)
print(f"Mejor R2 XGB: {study_xgb.best_value:.4f}")
joblib.dump(study_xgb, RUTA_CHECKPOINTS+'study_xgb.pkl')


In [ ]:
# ── EJERCICIO 8 — LightGBM con Optuna 

def objetivo_lgbm(trial):
    params = {
        # TU CODIGO: define el espacio de busqueda
        'random_state': SEED, 'n_jobs': -1, 'verbosity': -1
    }
    scores = cross_val_score(lgb.LGBMRegressor(**params), X_opt, y_opt,
                             cv=KFold(3,shuffle=False), scoring='r2', n_jobs=1)
    return scores.mean()

study_lgbm = optuna.create_study(direction='maximize',
               sampler=optuna.samplers.TPESampler(seed=SEED))
study_lgbm.optimize(objetivo_lgbm, n_trials=80, timeout=5400, show_progress_bar=True)
print(f"Mejor R2 LGBM: {study_lgbm.best_value:.4f}")
joblib.dump(study_lgbm, RUTA_CHECKPOINTS+'study_lgbm.pkl')


In [ ]:
# ── Entrenamiento final — celda completa ─────────────────────────────────────
def entrenar(nombre, Cls, params, Xtr, ytr, Xte, yte):
    print(f"Entrenando {nombre}...")
    m = Cls(**params); m.fit(Xtr, ytr); pred = m.predict(Xte)
    r2=r2_score(yte,pred); rmse=np.sqrt(mean_squared_error(yte,pred)); mae=mean_absolute_error(yte,pred)
    print(f"  {nombre}: R2={r2:.4f} | RMSE={rmse:.2f} | MAE={mae:.2f}")
    joblib.dump(m, RUTA_CHECKPOINTS+f'modelo_{nombre}.pkl')
    return m, pred, {'r2':r2,'rmse':rmse,'mae':mae}

p_rf   = {**study_rf.best_params,   'random_state':SEED,'n_jobs':-1}
p_xgb  = {**study_xgb.best_params,  'random_state':SEED,'n_jobs':-1,'tree_method':'hist'}
p_lgbm = {**study_lgbm.best_params, 'random_state':SEED,'n_jobs':-1,'verbosity':-1}

m_rf,   pred_rf,   res_rf   = entrenar('RF',   RandomForestRegressor, p_rf,   X_train,y_train,X_test,y_test)
m_xgb,  pred_xgb,  res_xgb  = entrenar('XGB',  xgb.XGBRegressor,      p_xgb,  X_train,y_train,X_test,y_test)
m_lgbm, pred_lgbm, res_lgbm = entrenar('LGBM', lgb.LGBMRegressor,     p_lgbm, X_train,y_train,X_test,y_test)


---
## Seccion 10 · Stacking con Red Neuronal — Ejercicio 9

**Misma arquitectura que el supermercado:**
```
Input(3) → Dense(64) → BatchNorm → Dropout(0.3)
         → Dense(32) → BatchNorm → Dropout(0.2)
         → Dense(16)
         → Dense(1)
```
Usa `EarlyStopping(patience=15)` y `ReduceLROnPlateau`.


In [ ]:
# ── EJERCICIO 9 — Red neuronal meta-aprendiz ─────────────────────────────────

X_mtr = np.column_stack([m_rf.predict(X_train), m_xgb.predict(X_train), m_lgbm.predict(X_train)])
X_mte = np.column_stack([pred_rf, pred_xgb, pred_lgbm])
sc_m  = StandardScaler()
Xmtr_s= sc_m.fit_transform(X_mtr)
Xmte_s= sc_m.transform(X_mte)
ytr_a = y_train.values.reshape(-1, 1)

# TU CODIGO: construye y entrena la red ──────────────────────────────────────
inp = keras.Input(shape=(3,))
# ... agrega las capas Dense, BatchNorm, Dropout ...
out = ___   # TU CODIGO: capa de salida

red = keras.Model(inp, out)
red.compile(optimizer=___, loss=___, metrics=['mae'])

cbs = [
    callbacks.EarlyStopping(patience=___, restore_best_weights=True),
    callbacks.ModelCheckpoint(RUTA_CHECKPOINTS+'red_ropa.keras', save_best_only=True),
    callbacks.ReduceLROnPlateau(patience=7, factor=0.5)
]

historia = red.fit(Xmtr_s, ytr_a, validation_split=___, epochs=___, batch_size=4096,
                   callbacks=cbs, verbose=1)

pred_meta = red.predict(Xmte_s).flatten()
r2_meta   = r2_score(y_test, pred_meta)
rmse_meta = np.sqrt(mean_squared_error(y_test, pred_meta))
mae_meta  = mean_absolute_error(y_test, pred_meta)
print(f"Stacking NN: R2={r2_meta:.4f} | RMSE={rmse_meta:.2f} | MAE={mae_meta:.2f}")


---
## Seccion 11 · Sistema de Decision — Ejercicio 10

### Sistema de decision para tienda de ropa

A diferencia del supermercado, en moda hay una accion adicional: **liquidar**.
Un inventario que no rota pierde valor rapidamente (la moda cambia de temporada).

**Reglas a implementar:**

| Condicion                                                | Accion              |
|----------------------------------------------------------|---------------------|
| `stock < 0.40 * demanda_7d`                             | `REORDENAR_URGENTE` |
| `stock < demanda_7d`                                     | `REORDENAR_NORMAL`  |
| `stock > 1.80 * demanda_7d` Y `dias_en_tienda > 30`    | `LIQUIDAR`          |
| `stock > demanda_7d` (pero reciente)                    | `ESPERAR`           |
| En cualquier otro caso                                   | `OK`                |

**Parametro adicional:** `dias_en_tienda` — cuantos dias lleva ese lote sin venderse.

**Casos de prueba:**

| Tienda        | Categoria    | Demanda 7d | Stock | Dias en tienda |
|---------------|--------------|-----------|-------|----------------|
| R01 El Tesoro | Camisetas    | 120       | 28    | 5              |
| R04 Unico     | Pantalones   | 85        | 45    | 15             |
| R02 Santafe   | Vestidos     | 60        | 180   | 45             |
| R03 Los Moln. | Accesorios   | 40        | 55    | 10             |
| R06 Mayorca   | Calzado      | 90        | 88    | 8              |


In [ ]:
# ── EJERCICIO 10 — Sistema de decision de ropa ───────────────────────────────

def decision_ropa(tienda, categoria, demanda_7d, stock_actual, dias_en_tienda):
    """
    Retorna dict con 'accion' y 'mensaje'.
    """
    # TU CODIGO ────────────────────────────────────────────────────────────────

    if ___:
        accion  = 'REORDENAR_URGENTE'
        mensaje = ___

    elif ___:
        accion  = 'REORDENAR_NORMAL'
        mensaje = ___

    elif ___ and ___:       # exceso + viejo
        accion  = 'LIQUIDAR'
        pct_rec = 30 if dias_en_tienda < 60 else 50
        mensaje = ___

    elif ___:               # exceso reciente
        accion  = 'ESPERAR'
        mensaje = ___

    else:
        accion  = 'OK'
        mensaje = f"[OK] {tienda} - {categoria}: niveles adecuados."

    return {'accion': accion, 'mensaje': mensaje}

# Casos de prueba
casos = [
    ('R01 El Tesoro',   'Camisetas y Tops', 120, 28,  5),
    ('R04 Unico',       'Pantalones',        85, 45, 15),
    ('R02 Santafe',     'Vestidos',          60, 180, 45),
    ('R03 Los Molinos', 'Accesorios',        40, 55, 10),
    ('R06 Mayorca',     'Calzado',           90, 88,  8),
]
print("SISTEMA DE DECISION — TIENDA DE ROPA")
print("=" * 65)
for args in casos:
    rec = decision_ropa(*args)
    print(rec['mensaje'])


---
## Tabla comparativa y preguntas de reflexion — Ejercicio 11

### Tabla final

Completa la tabla con todos los resultados obtenidos.


In [ ]:
# ── EJERCICIO 11 — Tabla final ───────────────────────────────────────────────

print("COMPARACION FINAL")
print("=" * 55)
print(f"{'Modelo':<20} {'RMSE':>8} {'MAE':>8} {'R2':>8}")
print("-" * 55)

todos = [
    ('Baseline Ridge',  RESULTADOS['baseline']['rmse'], RESULTADOS['baseline']['mae'], RESULTADOS['baseline']['r2']),
    ('Random Forest',   ___, ___, ___),   # TU CODIGO
    ('XGBoost',         ___, ___, ___),
    ('LightGBM',        ___, ___, ___),
    ('Stacking NN',     rmse_meta, mae_meta, r2_meta),
]
for nombre, rmse, mae, r2 in todos:
    print(f"{nombre:<20} {rmse:>8.2f} {mae:>8.2f} {r2:>8.4f}")
print("=" * 55)

# Feature importance
importancias = pd.Series(m_lgbm.feature_importances_, index=FEATURES).sort_values()
importancias.plot(kind='barh', figsize=(9,5), color='steelblue', edgecolor='white')
plt.title('Feature Importance — LightGBM (Ropa)')
plt.tight_layout(); plt.show()


### Preguntas de reflexion — responde en una celda Markdown

1. **Estacionalidad:** ¿En qué mes se concentra el pico de ventas?
   ¿Es mayo (Dia de la Madre) o enero/julio (liquidacion)?
   ¿Que dice eso sobre el peso del precio vs el volumen?

2. **Feature importance:** ¿Aparece `temporada_moda` o `es_dia_madre`
   en el top-5? Si no aparece, ¿qué explicacion le das?

3. **Liquidacion vs demanda:** Durante liquidacion las ventas en
   unidades suben pero el precio baja. ¿Tendria mas sentido usar
   `ingresos` como target en lugar de `items_dia`?

4. **Diferencia entre CC:** ¿Deberia haber un modelo por CC
   (El Tesoro vs Unico son perfiles muy diferentes)
   o un modelo unico tiene suficiente poder?

5. **Stacking:** ¿La red neuronal mejoro sobre el mejor ensemble?
   Si no: ¿data leakage en train o simplemente el ensemble ya
   capto toda la informacion disponible?
